# 02 — Data Cleaning

## Customer Behaviour & Sales Intelligence

### Objective
Prepare the Online Retail dataset for reliable sales analysis
and customer segmentation.

### Data Cleaning Plan
- Handle missing values
- Identify and remove duplicate records
- Handle cancelled transactions
- Check invalid quantities and prices
- Create a clean dataset for further analysis

### Important
We will preserve the original dataset and perform cleaning
on a separate DataFrame.

In [1]:
import pandas as pd
import numpy as np

# Load the original retail dataset
df = pd.read_excel("../data/Online Retail.xlsx")

# Check dataset dimensions
print("Dataset shape:", df.shape)

Dataset shape: (541909, 8)


In [3]:
# separate copy for data cleaning
df_clean = df.copy()

# Verify the copied dataset
print("Original dataset:", df.shape)
print("Cleaning dataset:", df_clean.shape)

Original dataset: (541909, 8)
Cleaning dataset: (541909, 8)


In [4]:
# Check missing values in each column
missing_values = df_clean.isnull().sum()

print("Missing Values by Column:")
print(missing_values)

Missing Values by Column:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [5]:
# Calculate missing value percentage for each column
missing_percentage = (df_clean.isnull().sum() / len(df_clean)) * 100

print("Missing Values Percentage:")
print(missing_percentage.round(2))

Missing Values Percentage:
InvoiceNo       0.00
StockCode       0.00
Description     0.27
Quantity        0.00
InvoiceDate     0.00
UnitPrice       0.00
CustomerID     24.93
Country         0.00
dtype: float64


In [6]:
# Create a dataset containing only transactions with valid CustomerID
df_customers = df_clean.dropna(subset=["CustomerID"]).copy()

# Compare dataset sizes
print("Total transactions:", len(df_clean))
print("Transactions with CustomerID:", len(df_customers))
print("Transactions without CustomerID:", len(df_clean) - len(df_customers))

Total transactions: 541909
Transactions with CustomerID: 406829
Transactions without CustomerID: 135080


In [7]:
# Check duplicate records
duplicate_count = df_clean.duplicated().sum()

print("Total duplicate records:", duplicate_count)

Total duplicate records: 5268


In [8]:
# Display duplicate records for inspection
duplicate_rows = df_clean[
    df_clean.duplicated(keep=False)
]

print("Total rows involved in duplicate groups:", len(duplicate_rows))

display(
    duplicate_rows.sort_values(
        by=["InvoiceNo", "StockCode"]
    ).head(10)
)

Total rows involved in duplicate groups: 10147


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,17920.0,United Kingdom
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920.0,United Kingdom


In [9]:
# Inspect one group of identical duplicate records
sample_duplicate = duplicate_rows.iloc[0]

matching_rows = df_clean[
    df_clean.eq(sample_duplicate).all(axis=1)
]

display(matching_rows)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom


### Duplicate Records — Initial Assessment

- The dataset contains 5,268 duplicate rows.
- A sample duplicate group contains 2 identical records.
- Identical rows may represent data-entry duplication or genuine repeated purchases.
- We will retain the original records for now and investigate further before removing duplicates.
- This approach avoids accidentally removing valid sales transactions.

In [10]:
# Identify cancelled invoices (invoice numbers starting with C)
cancelled_mask = df_clean["InvoiceNo"].astype(str).str.startswith("C")

# Count cancelled transaction rows
cancelled_count = cancelled_mask.sum()

print("Cancelled transaction rows:", cancelled_count)
print("Cancelled percentage:", round(cancelled_count / len(df_clean) * 100, 2), "%")

Cancelled transaction rows: 9288
Cancelled percentage: 1.71 %


In [11]:
# Display first 10 cancelled transaction rows
cancelled_transactions = df_clean[cancelled_mask].copy()

display(
    cancelled_transactions[
        ["InvoiceNo", "StockCode", "Quantity", "UnitPrice", "CustomerID"]
    ].head(10)
)

,InvoiceNo,StockCode,Quantity,UnitPrice,CustomerID
141,C536379,D,-1,27.50,14527.0
154,C536383,35004C,-1,4.65,15311.0
235,C536391,22556,-12,1.65,17548.0
236,C536391,21984,-24,0.29,17548.0
237,C536391,21983,-24,0.29,17548.0
238,C536391,21980,-24,0.29,17548.0
239,C536391,21484,-12,3.45,17548.0
240,C536391,22557,-12,1.65,17548.0
241,C536391,22553,-24,1.65,17548.0
939,C536506,22960,-6,4.25,17897.0


In [12]:
# Separate cancelled and non-cancelled transactions
df_cancelled = df_clean[cancelled_mask].copy()
df_non_cancelled = df_clean[~cancelled_mask].copy()

print("Cancelled transaction rows:", len(df_cancelled))
print("Non-cancelled transaction rows:", len(df_non_cancelled))
print("Total rows:", len(df_cancelled) + len(df_non_cancelled))

Cancelled transaction rows: 9288
Non-cancelled transaction rows: 532621
Total rows: 541909


In [13]:
# Check negative and zero quantities in non-cancelled transactions
negative_qty = (df_non_cancelled["Quantity"] < 0).sum()
zero_qty = (df_non_cancelled["Quantity"] == 0).sum()

print("Negative quantity rows:", negative_qty)
print("Zero quantity rows:", zero_qty)

Negative quantity rows: 1336
Zero quantity rows: 0


In [14]:
# Inspect negative quantities in non-cancelled invoices
negative_qty_rows = df_non_cancelled[
    df_non_cancelled["Quantity"] < 0
]

display(
    negative_qty_rows[
        ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"]
    ].head(10)
)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice
2406,536589,21777,NaN,-10,0.0
4347,536764,84952C,NaN,-38,0.0
7188,536996,22712,NaN,-20,0.0
7189,536997,22028,NaN,-20,0.0
7190,536998,85067,NaN,-6,0.0
7192,537000,21414,NaN,-22,0.0
7193,537001,21653,NaN,-6,0.0
7195,537003,85126,NaN,-2,0.0
7196,537004,21814,NaN,-30,0.0
7197,537005,21692,NaN,-70,0.0


In [15]:
# Check missing descriptions in negative-quantity rows
missing_desc = negative_qty_rows["Description"].isna().sum()

print("Total negative quantity rows:", len(negative_qty_rows))
print("Missing descriptions:", missing_desc)

Total negative quantity rows: 1336
Missing descriptions: 862


In [16]:
# Inspect negative-quantity rows with available descriptions
described_negative_rows = negative_qty_rows.dropna(
    subset=["Description"]
)

display(
    described_negative_rows[
        ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"]
    ].head(10)
)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice
7313,537032,21275,?,-30,0.0
13217,537425,84968F,check,-20,0.0
13218,537426,84968E,check,-35,0.0
13264,537432,35833G,damages,-43,0.0
21338,538072,22423,faulty,-13,0.0
21518,538090,20956,?,-723,0.0
22296,538161,46000S,Dotcom sales,-100,0.0
22297,538162,46000M,Dotcom sales,-100,0.0
42564,540010,22501,reverse 21/5/10 adjustment,-100,0.0
42566,540012,22502,reverse 21/5/10 adjustment,-100,0.0


In [17]:
# Check prices for negative-quantity transactions
print("Zero price rows:", (negative_qty_rows["UnitPrice"] == 0).sum())
print("Positive price rows:", (negative_qty_rows["UnitPrice"] > 0).sum())
print("Negative price rows:", (negative_qty_rows["UnitPrice"] < 0).sum())

Zero price rows: 1336
Positive price rows: 0
Negative price rows: 0


In [18]:
# Keep only non-cancelled transactions
# with positive quantity and positive unit price

df_sales = df_non_cancelled[
    (df_non_cancelled["Quantity"] > 0) &
    (df_non_cancelled["UnitPrice"] > 0)
].copy()

print("Valid sales rows:", len(df_sales))
print("Excluded from non-cancelled data:",
      len(df_non_cancelled) - len(df_sales))

Valid sales rows: 530104
Excluded from non-cancelled data: 2517


In [19]:
# Verify the sales dataset
print("Total valid sales rows:", len(df_sales))
print("Invalid quantities:", (df_sales["Quantity"] <= 0).sum())
print("Invalid prices:", (df_sales["UnitPrice"] <= 0).sum())
print("Cancelled invoices:", df_sales["InvoiceNo"].astype(str).str.startswith("C").sum())

Total valid sales rows: 530104
Invalid quantities: 0
Invalid prices: 0
Cancelled invoices: 0


In [20]:
# Calculate revenue for each sales transaction
df_sales["Revenue"] = df_sales["Quantity"] * df_sales["UnitPrice"]

# Preview the results
display(
    df_sales[
        ["InvoiceNo", "Quantity", "UnitPrice", "Revenue"]
    ].head(10)
)

,InvoiceNo,Quantity,UnitPrice,Revenue
0,536365,6,2.55,15.30
1,536365,6,3.39,20.34
2,536365,8,2.75,22.00
3,536365,6,3.39,20.34
4,536365,6,3.39,20.34
5,536365,2,7.65,15.30
6,536365,6,4.25,25.50
7,536366,6,1.85,11.10
8,536366,6,1.85,11.10
9,536367,32,1.69,54.08


In [21]:
# Calculate total sales revenue
total_revenue = df_sales["Revenue"].sum()

print(f"Total Sales Revenue: £{total_revenue:,.2f}")

Total Sales Revenue: £10,666,684.54


In [22]:
# Count unique customers in valid sales
unique_customers = df_sales["CustomerID"].nunique()

print("Unique Customers:", unique_customers)

Unique Customers: 4338


In [23]:
# Check missing CustomerID in valid sales
missing_customers = df_sales["CustomerID"].isna().sum()

print("Sales rows without CustomerID:", missing_customers)
print("Sales rows with CustomerID:", len(df_sales) - missing_customers)

Sales rows without CustomerID: 132220
Sales rows with CustomerID: 397884


In [24]:
# Create customer-level analysis dataset
df_customer_sales = df_sales.dropna(
    subset=["CustomerID"]
).copy()

print("Customer sales rows:", len(df_customer_sales))
print("Unique customers:", df_customer_sales["CustomerID"].nunique())
print("Missing CustomerIDs:", df_customer_sales["CustomerID"].isna().sum())

Customer sales rows: 397884
Unique customers: 4338
Missing CustomerIDs: 0


In [25]:
# Check duplicate rows in the valid sales dataset
sales_duplicates = df_sales.duplicated().sum()

print("Duplicate rows in valid sales:", sales_duplicates)

Duplicate rows in valid sales: 5226


In [26]:
# Compare revenue with and without exact duplicate rows

revenue_with_duplicates = df_sales["Revenue"].sum()

revenue_without_duplicates = (
    df_sales.drop_duplicates()["Revenue"].sum()
)

revenue_difference = (
    revenue_with_duplicates - revenue_without_duplicates
)

print(f"Revenue with duplicates: £{revenue_with_duplicates:,.2f}")
print(f"Revenue without duplicates: £{revenue_without_duplicates:,.2f}")
print(f"Revenue difference: £{revenue_difference:,.2f}")

Revenue with duplicates: £10,666,684.54
Revenue without duplicates: £10,642,110.80
Revenue difference: £24,573.74


### Duplicate Records — Revenue Impact

- Exact duplicate rows in valid sales: 5,226
- Preliminary revenue: £10,666,684.54
- Revenue after removing exact duplicates: £10,642,111.54
- Potential revenue difference: £24,573.00

**Decision:** Retain duplicates for now because identical
retail transaction rows may represent genuine purchases.
The revenue impact has been quantified for transparency.
Further validation is required before permanently removing them.

In [27]:
# Save valid sales dataset for further analysis
df_sales.to_csv("../data/cleaned_sales.csv", index=False)

print("Cleaned sales dataset saved successfully!")

Cleaned sales dataset saved successfully!


In [28]:
# Save customer dataset for segmentation
df_customer_sales.to_csv(
    "../data/customer_sales.csv",
    index=False
)

print("Customer dataset saved successfully!")
print("Total rows:", len(df_customer_sales))
print("Unique customers:", df_customer_sales["CustomerID"].nunique())

Customer dataset saved successfully!
Total rows: 397884
Unique customers: 4338
